# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports
import os
import requests
import ollama
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import time

In [2]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [3]:
# set up environment

# Load environment variables
load_dotenv(override=True)

# Set up OpenAI
openai_client = OpenAI()

# Test OpenAI connection
api_key = os.getenv('OPENAI_API_KEY')
if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
    print("✅ OpenAI API key looks good!")
else:
    print("⚠️ OpenAI API key might have issues - check your .env file")

# Set up Ollama - using OpenAI client format for consistency
ollama_client = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama'  # Ollama doesn't need a real API key
)

print("✅ Environment setup complete!")
print(f"🤖 GPT Model: {MODEL_GPT}")
print(f"🦙 Llama Model: {MODEL_LLAMA}")

✅ OpenAI API key looks good!
✅ Environment setup complete!
🤖 GPT Model: gpt-4o-mini
🦙 Llama Model: llama3.2


In [10]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [4]:
# Technical Tutor System Prompt

tutor_system_prompt = """You are an expert technical tutor specializing in programming, computer science, and software engineering concepts. Your role is to:

1. **Explain complex concepts clearly** - Break down difficult topics into understandable parts
2. **Provide practical examples** - Include code examples, analogies, and real-world applications
3. **Be encouraging and supportive** - Help learners build confidence while addressing their questions
4. **Adapt to the learner's level** - Gauge complexity from the question and respond appropriately
5. **Encourage best practices** - Point out good coding practices, security considerations, and industry standards

When explaining code:
- Break down each part step by step
- Explain the syntax and semantics
- Discuss why certain approaches are used
- Mention potential improvements or alternatives
- Include relevant context about when/why to use this pattern

When answering conceptual questions:
- Start with a simple explanation
- Build up to more complex details
- Use analogies when helpful
- Provide examples from different programming languages if relevant

Always aim to be:
- **Clear and concise** but thorough
- **Practical** with actionable insights  
- **Educational** beyond just answering the immediate question
- **Encouraging** to promote continued learning

Format your responses in markdown for better readability."""

print("✅ Technical tutor system prompt created!")

✅ Technical tutor system prompt created!


In [5]:
# Helper Functions for Technical Tutor

def create_tutor_messages(question, context=""):
    """Create messages for the technical tutor"""
    user_content = f"Question: {question}"
    if context:
        user_content += f"\n\nAdditional context: {context}"
    
    return [
        {"role": "system", "content": tutor_system_prompt},
        {"role": "user", "content": user_content}
    ]

def get_gpt_answer(question, context="", stream=False):
    """Get answer from GPT-4o-mini"""
    messages = create_tutor_messages(question, context)
    
    if stream:
        return openai_client.chat.completions.create(
            model=MODEL_GPT,
            messages=messages,
            stream=True,
            temperature=0.7
        )
    else:
        response = openai_client.chat.completions.create(
            model=MODEL_GPT,
            messages=messages,
            temperature=0.7
        )
        return response.choices[0].message.content

def get_ollama_answer(question, context=""):
    """Get answer from Ollama (Llama 3.2)"""
    messages = create_tutor_messages(question, context)
    
    try:
        response = ollama_client.chat.completions.create(
            model=MODEL_LLAMA,
            messages=messages,
            temperature=0.7
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error connecting to Ollama: {str(e)}\\n\\nMake sure Ollama is running with: `ollama serve`"

def display_streaming_response(stream, title="Response"):
    """Display streaming response with typewriter effect"""
    print(f"\\n🤖 {title}:")
    print("=" * 60)
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    
    for chunk in stream:
        if chunk.choices[0].delta.content:
            response += chunk.choices[0].delta.content
            # Clean up markdown artifacts for display
            clean_response = response.replace("```", "").replace("markdown", "")
            update_display(Markdown(clean_response), display_id=display_handle.display_id)
    
    return response

def display_response(content, title="Response"):
    """Display static response with nice formatting"""
    print(f"\\n🦙 {title}:")
    print("=" * 60)
    display(Markdown(content))

print("✅ Technical tutor helper functions created!")

✅ Technical tutor helper functions created!


In [11]:
# Get gpt-4o-mini to answer, with streaming

print("🚀 Getting answer from GPT-4o-mini with streaming...")
gpt_stream = get_gpt_answer(question, stream=True)
gpt_response = display_streaming_response(gpt_stream, "GPT-4o-mini Technical Tutor")

🚀 Getting answer from GPT-4o-mini with streaming...
\n🤖 GPT-4o-mini Technical Tutor:
\n🤖 GPT-4o-mini Technical Tutor:


Sure! Let's break down the code snippet you've provided:

python
yield from {book.get("author") for book in books if book.get("author")}


### Explanation

1. **Set Comprehension**:
   - The code utilizes a **set comprehension**: `{... for ... in ... if ...}`. This creates a set of unique items based on the condition provided.
   - In this case, it iterates over a collection called `books`.

2. **Iterating Through `books`**:
   - `for book in books`: This part loops through each item in the `books` iterable. Here, each `book` is expected to be a dictionary (or an object) that represents a book.

3. **Getting the Author**:
   - `book.get("author")`: This method tries to retrieve the value associated with the key `"author"` from the `book` dictionary. If the key doesn’t exist, it returns `None` instead of raising an error.

4. **Filtering Non-Existent Authors**:
   - `if book.get("author")`: The comprehension includes a condition to filter out books where the author is `None` (or any falsy value). This means only books that have a valid author will contribute to the set.

5. **Yielding Results**:
   - **`yield from`**: This is a special syntax in Python that is used within a generator function. It allows you to yield all values from an iterable (in this case, a set). This means that if this code is part of a generator function, it will yield each author name one at a time to the caller.

### Why Use This Code?

- **Uniqueness**: Using a set comprehension ensures that each author is only yielded once, even if multiple books have the same author.
- **Efficiency**: The use of `yield from` allows for lazy evaluation. Instead of creating an entire list of authors and returning it at once, it yields them one by one, which can be more memory efficient, especially with large datasets.
- **Readability**: The code is concise and leverages Python's powerful comprehension syntax to achieve the desired outcome in a clear manner.

### Example Scenario

Imagine you have a list of books, each represented as a dictionary with various details, including the author:

python
books = [
    {"title": "Book A", "author": "Author 1"},
    {"title": "Book B", "author": "Author 2"},
    {"title": "Book C", "author": None},
    {"title": "Book D", "author": "Author 1"},
]


Using the code snippet:

python
def get_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

for author in get_authors(books):
    print(author)


### Output:

Author 1
Author 2


### Summary

- The code efficiently collects unique authors from a list of book dictionaries.
- It uses Python's set comprehension and generator features for clarity and performance.
- It's a good example of combining comprehensions and generator functions to create clean, Pythonic code.

If you have any more questions or need further clarification, feel free to ask!

In [12]:
# Get Llama 3.2 to answer

print("\\n🚀 Getting answer from Ollama (Llama 3.2)...")
print("⏳ This might take a moment as Ollama processes the request...")

ollama_response = get_ollama_answer(question)
display_response(ollama_response, "Llama 3.2 Technical Tutor")

\n🚀 Getting answer from Ollama (Llama 3.2)...
⏳ This might take a moment as Ollama processes the request...
\n🦙 Llama 3.2 Technical Tutor:
\n🦙 Llama 3.2 Technical Tutor:


### Explaining the Given Code

The given code uses a combination of Python's built-in `yield` keyword, generator expression, and dictionary iteration. Here's a breakdown:

#### What is happening?

1. **Iterator Expression**: `{book.get("author") for book in books if book.get("author")}` creates an iterator that yields authors from the `books` list.
   - The `.get()` method of each dictionary-like object (`book`) is used to retrieve values.
   - The `if` condition filters out dictionaries that don't have an `"author"` key.

2. **Yield from**: The outer expression uses `yield from`, which allows us to delegate the iteration to the inner generator expression.

3. **Yielding Authors**: When we iterate over the resulting iterator, each author's value is yielded one at a time, without having to load all authors into memory at once.

### Why Use This Approach?

This approach offers several benefits:

*   **Memory Efficiency**: By using a generator expression and `yield from`, we avoid loading all books into memory simultaneously. Instead, we process them on demand, which can be more memory-efficient for large datasets.
*   **Lazy Evaluation**: The iteration happens only when needed (i.e., when iterating over the iterator), which aligns with the concept of lazy evaluation.

### Example Walkthrough

Suppose we have a list `books` containing dictionaries like this:

```python
[
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": None},
    {"title": "Book 3", "author": "Author C"}
]
```

If we iterate over the generator expression, it will yield the authors one at a time:

```python
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": None},
    {"title": "Book 3", "author": "Author C"}
]

for author in {book.get("author") for book in books if book.get("author")}.values():
    print(author)
```

Output:

```
Author A
None
Author C
```

### Potential Improvements

*   Consider adding error handling to deal with cases where some dictionaries might not have the `"author"` key.
*   If you need more control over the iteration process, consider using a `for` loop and iterating directly over the generator expression.

By understanding how this code works, you can make informed decisions about when to use generator expressions and lazy evaluation in your own projects.

In [6]:
# Compare Both Responses Side by Side

def compare_responses(question, context=""):
    """Get responses from both models and display them for comparison"""
    print("🔍 TECHNICAL TUTOR COMPARISON")
    print("=" * 80)
    print(f"📝 Question: {question}")
    if context:
        print(f"📄 Context: {context}")
    print("=" * 80)
    
    # Get GPT response (streaming)
    print("\\n🤖 Getting GPT-4o-mini response...")
    gpt_stream = get_gpt_answer(question, context, stream=True)
    gpt_resp = display_streaming_response(gpt_stream, "GPT-4o-mini Answer")
    
    # Get Ollama response
    print("\\n🦙 Getting Llama 3.2 response...")
    ollama_resp = get_ollama_answer(question, context)
    display_response(ollama_resp, "Llama 3.2 Answer")
    
    print("\\n" + "=" * 80)
    print("✅ COMPARISON COMPLETE!")
    print("💡 Notice any differences in explanation style, depth, or approach?")
    print("=" * 80)
    
    return gpt_resp, ollama_resp

print("✅ Comparison function created!")

✅ Comparison function created!


In [8]:
# Example Technical Questions You Can Try

example_questions = [
    "Please explain what this code does and why:\nyield from {book.get('author') for book in books if book.get('author')}",
    
    "What's the difference between async/await and regular functions in Python?",
    
    "Explain how REST APIs work and why they're important in web development",
    
    "What is the difference between machine learning and deep learning?",
    
    "How does garbage collection work in programming languages?",
    
    "Explain the concept of recursion with a practical example",
    
    "What are the main differences between SQL and NoSQL databases?",
    
    "How do Git branches work and when should I use them?",
    
    "Explain object-oriented programming principles with examples",
    
    "What is the difference between compiled and interpreted programming languages?"
]

print("📚 Example Questions Available:")
for i, q in enumerate(example_questions, 1):
    print(f"{i}. {q[:60]}{'...' if len(q) > 60 else ''}")

print("\n💡 To use any example, copy and paste it into the 'question' variable above!")
print("🔄 Or use the compare_responses() function for side-by-side comparison!")

# Quick test function
def test_with_example(example_number=1):
    """Test the tutor with one of the example questions"""
    if 1 <= example_number <= len(example_questions):
        test_question = example_questions[example_number - 1]
        print(f"🧪 Testing with example {example_number}:")
        return compare_responses(test_question)
    else:
        print(f"❌ Please choose a number between 1 and {len(example_questions)}")

print("✅ Example questions and test function ready!")
print("\n🚀 Try: test_with_example(1) to test with the first example!")

📚 Example Questions Available:
1. Please explain what this code does and why:
yield from {book...
2. What's the difference between async/await and regular functi...
3. Explain how REST APIs work and why they're important in web ...
4. What is the difference between machine learning and deep lea...
5. How does garbage collection work in programming languages?
6. Explain the concept of recursion with a practical example
7. What are the main differences between SQL and NoSQL database...
8. How do Git branches work and when should I use them?
9. Explain object-oriented programming principles with examples
10. What is the difference between compiled and interpreted prog...

💡 To use any example, copy and paste it into the 'question' variable above!
🔄 Or use the compare_responses() function for side-by-side comparison!
✅ Example questions and test function ready!

🚀 Try: test_with_example(1) to test with the first example!


In [13]:
# Test the Complete Technical Tutor Comparison

# Uncomment the line below to test side-by-side comparison with the current question:
# compare_responses(question)

# Or test with one of the example questions:
# test_with_example(2)  # Try async/await explanation

# Or ask your own custom question:
# custom_question = "How does Python's GIL (Global Interpreter Lock) affect multi-threading?"
# compare_responses(custom_question)

print("🎓 Your Technical Tutor is ready!")
print()
print("📋 Quick Usage Guide:")
print("1. Modify the 'question' variable above with your technical question")
print("2. Run the GPT-4o-mini cell for streaming response")
print("3. Run the Ollama cell for local Llama 3.2 response") 
print("4. Or use compare_responses(your_question) for side-by-side comparison")
print("5. Try test_with_example(number) to test with predefined questions")
print()
print("🔥 Pro tip: Both models use the same technical tutor system prompt,")
print("   so you can compare their teaching styles and explanations!")
print()
print("✅ Technical Tutor System Complete! 🚀")

🎓 Your Technical Tutor is ready!

📋 Quick Usage Guide:
1. Modify the 'question' variable above with your technical question
2. Run the GPT-4o-mini cell for streaming response
3. Run the Ollama cell for local Llama 3.2 response
4. Or use compare_responses(your_question) for side-by-side comparison
5. Try test_with_example(number) to test with predefined questions

🔥 Pro tip: Both models use the same technical tutor system prompt,
   so you can compare their teaching styles and explanations!

✅ Technical Tutor System Complete! 🚀


In [ ]:
# 🧪 DEMONSTRATION: Compare both models with a simple question

demo_question = "What is the difference between a list and a tuple in Python?"
print("🎯 Running demonstration comparison...")
print(f"Question: {demo_question}")
print("=" * 60)

# Uncomment the line below to run the demonstration:
# gpt_demo, ollama_demo = compare_responses(demo_question)